[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 06](README.md)

# CUDA: bibliotecas, streams y perfiles

**Tema:** 06 · **Sesiones:** 28, 29, 30 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cuándo usar una biblioteca acelerada y cómo separar preparación, transferencia y ejecución repetida?


## Resultados de aprendizaje

- Seleccionar Thrust/CUB o una biblioteca de dominio.
- Administrar handle, plan, descriptor, workspace y stream.
- Interpretar Nsight y tiempo extremo a extremo sin ocultar preparación.


## Modelo conceptual

Las bibliotecas aprovechan algoritmos y kernels especializados cuando layout, tipo y tamaño coinciden con su contrato.

Planes y workspaces se reutilizan; crearlos dentro de cada repetición distorsiona la medición.

Una comparación justa mantiene operación matemática, precisión, tolerancia, datos residentes y costos incluidos.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "06"
NOTEBOOK = "06_cuda/03_bibliotecas_perfiles.ipynb"
assert (ROOT / "curso" / "notebooks" / "06_cuda" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Selección inicial

Una función explícita evita recomendar una biblioteca sin considerar la operación.


In [ ]:
def candidate(operation):
    return {
        "transform": "Thrust",
        "reduce": "CUB/Thrust",
        "gemm": "cuBLAS/cuBLASLt",
        "fft": "cuFFT",
        "spmv": "cuSPARSE",
        "dense_solve": "cuSOLVER",
        "random": "cuRAND",
    }.get(operation, "kernel propio o composición")
operations = ("transform", "reduce", "gemm", "fft", "spmv", "dense_solve", "random", "stencil_fused")
assert candidate("gemm") == "cuBLAS/cuBLASLt"
assert candidate("stencil_fused") == "kernel propio o composición"
for operation in operations: print(f"{operation:14} -> {candidate(operation)}")


**Interpretación.** La selección final incorpora tamaños, layout, residencia, precisión, repetición y posibilidad de fusión.


## Perfil de fases

Se resume una serie repetida mediante mediana y se separa preparación.


In [ ]:
import statistics
phases = {
    "plan_once": [3.4],
    "h2d": [1.1, 1.0, 1.2, 1.1, 1.0],
    "compute": [0.42, 0.40, 0.41, 0.43, 0.40],
    "d2h": [0.8, 0.82, 0.79, 0.81, 0.8],
}
medians = {name: statistics.median(values) for name, values in phases.items()}
repeated_total = medians["h2d"] + medians["compute"] + medians["d2h"]
print(medians, "total_repetido_ms", repeated_total)
assert repeated_total > medians["compute"]


**Interpretación.** Se publican tanto ejecución con datos residentes como extremo a extremo; el plan se amortiza solo cuando se reutiliza.


## Práctica reproducible

1. Implementar el ciclo crear–configurar–ejecutar–validar–destruir.
2. Comparar una biblioteca con referencia y kernel correcto.
3. Conservar comandos Nsight y métricas que respondan la pregunta.


## Errores frecuentes

- Incluir creación de plan solo en una variante.
- Comparar column-major con row-major sin ajustar layout.
- Usar ocupación alta como sinónimo de rendimiento.

## Criterios de aceptación

- Estados de biblioteca comprobados.
- Recursos liberados y streams documentados.
- Preparación, ejecución residente y total separados.


## Referencias y material relacionado

- [Guía y bibliotecas CUDA](README.md)
- [Ejemplos CUDA](../../ejemplos/06_cuda/README.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 06](README.md)
